# Phase 6 — MCP Server (tools as a standalone service)

**This is a lighter, mostly-reading phase.** No new module for you to write — the focus is *understanding* a real MCP server.

**Goal**: So far your `query_retail` tool ran **in-process** — inside the notebook's own Python program (`create_sdk_mcp_server`). That's convenient but not how tools are shared in the real world. This phase packages the same tool as a **standalone MCP server**: a separate program the agent launches and talks to over **stdio** (standard input/output).

That standalone form is what lets tools be reused, shared, versioned, and deployed independently — it's how every third-party MCP server (GitHub's, Slack's, ...) works.

## Key concepts you'll learn

| Concept | Plain-English |
|---|---|
| **In-process MCP** | The tool runs *inside* your program. Phases 1-5. Simple, but not shareable. |
| **Standalone MCP server** | The tool is its *own program*. Phase 6. Reusable by any agent or app. |
| **stdio transport** | The agent launches the server as a subprocess and talks to it through stdin/stdout. |
| **The server is a thin wrapper** | `retail_server.py` doesn't re-implement anything — it just exposes `orchestrator/tools.py` over MCP. |

## Acceptance criteria
1. The agent connects to the **standalone** server (not in-process) and calls its tool
2. The answer matches the pandas ground truth
3. `Trace` exported to `traces/traces.jsonl`

## 1. Setup

In [ ]:
# Make the orchestrator/ library importable from notebooks/
import sys, os
from pathlib import Path

# Walk up to the orchestrator root (the folder containing 'data/retail.parquet').
# Idempotent: safe to re-run this cell any number of times.
ROOT = Path.cwd().resolve()
while not (ROOT / "data" / "retail.parquet").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
assert (ROOT / "data" / "retail.parquet").exists(), f"Could not find orchestrator root from {Path.cwd()}"
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv()

assert os.getenv("CLAUDE_CODE_OAUTH_TOKEN") or os.getenv("ANTHROPIC_API_KEY"), \
    "No auth token found in .env. Run `claude setup-token` and paste into .env."
print(f"[OK] Working dir: {os.getcwd()}")
print(f"[OK] Dev model: {os.getenv('CLAUDE_DEV_MODEL')}")

In [ ]:
import pandas as pd

from orchestrator.tools import get_retail_df
from orchestrator.observability import Trace, Timer
from orchestrator.agents import run_agent
from claude_agent_sdk import ClaudeAgentOptions

DEV_MODEL = os.getenv("CLAUDE_DEV_MODEL", "claude-haiku-4-5-20251001")

df = get_retail_df()
print(f"Loaded {len(df):,} rows")
print(f"Dev model: {DEV_MODEL}")

## 2. Ground truth

A simple question to test the plumbing: top 5 countries by revenue in 2011.

In [ ]:
df_2011 = df[(df["Quantity"] > 0) & (df["InvoiceDate"].dt.year == 2011)]

ground_truth = (
    df_2011.groupby("Country")["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)
print("Top 5 countries by revenue, 2011:")
print(ground_truth)

expected_countries = sorted(ground_truth["Country"])
print()
print("Expected countries in the answer:", expected_countries)

## 3. Read the standalone MCP server

The server lives at **`mcp_servers/retail_server.py`**. Open it (or run the next cell to print it) and read it — that's the main task of this phase.

Things to notice as you read:
- It uses **`FastMCP`** — the standard way to build an MCP server in Python.
- `@mcp.tool()` turns a plain function into an MCP tool. The **docstring** becomes the tool description; the **type hints** become the schema.
- The function body just calls `orchestrator.tools.query_retail` — the server is a **thin wrapper**, it re-implements nothing.
- `mcp.run()` at the bottom starts the server on the **stdio** transport.
- There are **no `print()` calls** — stdout is the protocol channel, so printing would corrupt it.

In [ ]:
server_code = Path("mcp_servers/retail_server.py").read_text()
print(server_code)

## 4. In-process vs standalone — the difference

| | In-process (Phases 1-5) | Standalone (Phase 6) |
|---|---|---|
| Built with | `create_sdk_mcp_server(...)` | a separate script + `FastMCP` |
| Where it runs | inside the notebook's process | its own subprocess |
| Transport | direct function calls | **stdio** (stdin/stdout) |
| Reusable by other apps? | no | **yes** — any MCP client can use it |
| Good for | quick prototyping | real, shareable, deployable tools |

Same tool, same `query_retail` logic. The only thing that changed is the **packaging** — and that's what makes it shareable.

## 5. Connect the agent to the standalone server

To use a standalone server, you describe it as a **stdio MCP server config**: tell the SDK what command to run and which script. The SDK then launches it as a subprocess and connects.

**TODO 1**: point the config at the server script.

In [ ]:
# A stdio MCP server config: the SDK launches `command args...` as a
# subprocess and speaks the MCP protocol to it over stdin/stdout.
stdio_retail_server = {
    "type": "stdio",
    "command": sys.executable,
    # ----- TODO 1: point the config at the server script -----
    "args": ["mcp_servers/retail_server.py"],
    # ---------------------------------------------------------
}

mcp_options = ClaudeAgentOptions(
    model=DEV_MODEL,
    system_prompt=(
        "You are a retail data analyst. Answer the question using the "
        "query_retail tool, and present the result as a clean markdown "
        "table. Never invent numbers - report exactly what the tool returns."
    ),
    mcp_servers={"retail": stdio_retail_server},
    allowed_tools=["mcp__retail__query_retail"],
    max_turns=5,
)
print("[OK] Agent configured to use the STANDALONE stdio MCP server")

## 6. Run it

**TODO 2**: write the question — ask for the top 5 countries by revenue in 2011.

When this runs, the SDK launches `retail_server.py` as a subprocess, the agent calls `query_retail` on it over stdio, and the result comes back. (It may take a few extra seconds the first time — that's the subprocess starting up.)

In [ ]:
# ----- TODO 2: write your question -----
QUESTION = "What were the top 5 countries by revenue in 2011?"
# ---------------------------------------

trace = Trace(model=DEV_MODEL, question=QUESTION)
with Timer() as t:
    run = await run_agent("DataAgent", QUESTION, mcp_options)

trace.input_tokens        = run.input_tokens
trace.output_tokens       = run.output_tokens
trace.cached_input_tokens = run.cached_input_tokens
trace.n_tool_calls        = run.n_tool_calls
trace.n_delegations       = 1
trace.latency_ms          = t.elapsed_ms
trace.answer              = run.answer
trace.compute_cost()

print("----- Agent answer (via standalone MCP server) -----")
print(run.answer)
print(f"\nTool calls: {trace.n_tool_calls}")

## 7. Verify (acceptance criteria)

In [ ]:
# ----- TODO 3: write the assertion -----
missing = []
for country in expected_countries:
    if country not in trace.answer:
        missing.append(country)
# ---------------------------------------

trace.passed = (missing == [])
print(f"Missing countries: {missing}")
print(f"Passed: {trace.passed}")
assert trace.passed, f"Answer missed: {missing}"

# Phase 6 acceptance: the standalone server actually got used
assert trace.n_tool_calls >= 1, "Expected the agent to call the standalone MCP tool at least once"

trace.append_jsonl()
print("\n[OK] Acceptance criteria met - standalone MCP server used, trace saved")

## 8. Quiz (~5 min — answer in a new markdown cell)

**TODO 4**: Answer in your own words.

1. **In-process vs standalone**: what is the practical difference between the in-process MCP server (Phases 1-5) and the standalone one here? Give one real advantage of the standalone form.

   In-process: the tool function lives in the same Python program as the agent — when the notebook dies, the tool dies. Standalone: the tool is a separate program launched as a subprocess, talking over stdio. **Advantage**: language and process independence. A Node.js app, a Go agent, the Claude Desktop client, even a future agent you haven't written yet — any MCP client on any machine can use the same server. The tool becomes a *service*, not a library import.

2. **The thin wrapper**: `retail_server.py` doesn't re-implement the query logic — it imports `orchestrator.tools`. Why is that a good design? What would be bad about copy-pasting the logic into the server?

   Single source of truth. The pandas logic in `tools.py` is tested, exercised by every notebook, and changes in one place. Copy-pasting it into the server would create *two* implementations that drift — fix the year filter in `tools.py`, forget to fix it in the server, and the in-process agent gets the right answer while the standalone server quietly returns stale data. Always keep the server as a thin adapter over a single underlying module.

3. **stdio**: the server talks to the agent over stdin/stdout. Why must the server never `print()` anything to stdout? What would happen if it did?

   stdout is the **protocol channel** — every byte the server writes there is interpreted as an MCP JSON-RPC message. A stray `print("debug: starting...")` injects malformed JSON into the stream, the client's parser chokes, the connection breaks. If you need logging, write to **stderr** (or a file) — stderr is free for human-readable noise.

4. **Reuse**: now that the tool is a standalone server, name one *other* app or agent (not this notebook) that could use it — and explain why standalone packaging makes that possible.

   The Streamlit capstone in Phase 10 — it's a completely separate process from any notebook, but it can spawn `retail_server.py` and use the same `query_retail` over MCP. Or: Claude Desktop, by adding the server to `claude_desktop_config.json`. Standalone packaging makes both possible because the contract is "launch this command, speak MCP on stdio" — no shared Python imports, no shared process, no coupling beyond the protocol.

## Phase 6 done when:
- [x] You've read `mcp_servers/retail_server.py` and understand it
- [x] TODO 1 (stdio config args) filled in
- [x] TODO 2 (your question) filled in
- [x] TODO 3 (assertion) filled in
- [x] TODO 4 (quiz) answered
- [x] Notebook runs top-to-bottom without errors
- [x] `traces/traces.jsonl` has a new line

Then ping me — we'll review, then move to Phase 7 (human-in-the-loop).